In [0]:
from pyspark.sql import functions as F

df_fixed = spark.read.table('cyntexa_dev.sales.sales_bronze')

# Filter out invalid rows FIRST (while columns are still strings)
df_fixed = df_fixed.filter(
    (F.col('total_amount').cast('double') > 0) &
    (F.col('total_amount').cast('double') < 1000000000000000000) &
    (F.col('customer_id').cast('long') > 0) &
    (F.col('order_id').cast('long') > 0) &
    (F.col('product_id') != 'null') &
    (F.col('product_id').isNotNull())
)

# THEN cast to final datatypes
df_fixed = df_fixed.withColumn('order_date', F.col('order_date').cast('date')) \
       .withColumn('total_amount', F.col('total_amount').cast('decimal(18,2)')) \
       .withColumn('customer_id', F.col('customer_id').cast('long')) \
       .withColumn('order_id', F.col('order_id').cast('long'))

df_fixed = df_fixed.dropDuplicates()
df_fixed = df_fixed.dropna()

df_fixed.write.mode('overwrite').saveAsTable('cyntexa_dev.sales.sales_silver')